In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

x = pd.read_csv("fifa_world_cup(2).csv")
x.head()
x.info()

<class 'pandas.DataFrame'>
RangeIndex: 1337 entries, 0 to 1336
Data columns (total 70 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   tournament_name                1337 non-null   str    
 1   stage_name                     1337 non-null   str    
 2   group_name                     1271 non-null   str    
 3   group_stage                    1337 non-null   int64  
 4   knockout_stage                 1337 non-null   int64  
 5   replayed                       1337 non-null   int64  
 6   replay                         1337 non-null   int64  
 7   match_date                     1337 non-null   str    
 8   match_time                     1337 non-null   str    
 9   stadium_name                   1271 non-null   str    
 10  city_name                      1270 non-null   str    
 11  country_name                   1337 non-null   str    
 12  team_name                      1337 non-null   str    
 13 

In [2]:
x = x.drop("tournament_name", axis=1)
x.isnull().sum()

stage_name          0
group_name         66
group_stage         0
knockout_stage      0
replayed            0
                   ..
yellow_cards_h1     0
yellow_cards_h2     0
red_cards_h1        0
red_cards_h2        0
outcome             0
Length: 69, dtype: int64

In [3]:
x.duplicated().sum()

x["outcome"].value_counts()

outcome
Loss    548
Win     527
Draw    262
Name: count, dtype: int64

In [4]:
for col in x.columns:
    print(col, ":", x[col].unique()[:10])

stage_name : <ArrowStringArray>
[       'group stage',     'quarter-finals',        'semi-finals',
  'third-place match',              'final', 'second group stage',
        'round of 16']
Length: 7, dtype: str
group_name : <ArrowStringArray>
[       'Group 1',        'Group 2',        'Group 3',              nan,
 'not applicable',        'Group 4',        'Group B',        'Group A',
        'Group 5',        'Group 6']
Length: 10, dtype: str
group_stage : [1 0]
knockout_stage : [0 1]
replayed : [0]
replay : [0]
match_date : <ArrowStringArray>
['1970-05-31', '1970-06-03', '1970-06-06', '1970-06-07', '1970-06-10',
 '1970-06-11', '1970-06-14', '1970-06-17', '1970-06-20', '1970-06-21']
Length: 10, dtype: str
match_time : <ArrowStringArray>
['12:00', '16:00', '17:00', '19:30', '18:00', '19:00', '13:45', '19:15',
 '16:45', '15:00']
Length: 10, dtype: str
stadium_name : <ArrowStringArray>
[      'Estadio Azteca',         'La Bombonera',      'Estadio Jalisco',
                    nan,     

In [5]:
x = x.drop([
    "stage_name", "group_name", "replayed", "replay",
    "match_date", "match_time", "stadium_name", "city_name",
    "country_name", "team_code",
    "opponent_name", "opponent_code",
    "extra_time", "result",
    "penalty_shootout", "penalties_for", "penalties_against",
    "yellow_cards", "red_cards", "possession", "shots",
    "shots_on_target", "passes_completed", "passes_attempted",
    "corners", "fouls", "goals_for", "goals_against", 
    "possession_h1", "possession_h2", "shots_h1", "shots_h2",
    "shots_on_target_h1", "shots_on_target_h2",
    "passes_completed_h1", "passes_completed_h2",
    "passes_attempted_h1", "passes_attempted_h2",
    "corners_h1", "corners_h2", "fouls_h1", "fouls_h2",
    "yellow_cards_h1", "yellow_cards_h2",
    "red_cards_h1", "red_cards_h2"
], axis=1)

In [6]:
Y = x["outcome"]
X = x.drop("outcome", axis=1)

X = pd.get_dummies(X, drop_first=True)
X = X.fillna(X.median(numeric_only=True))

X.head()

,group_stage,knockout_stage,home_team,away_team,is_host,team_prior_matches,team_prior_win_rate,team_prior_goals_scored_avg,team_prior_goals_conceded_avg,opp_prior_matches,...,team_name_Tunisia,team_name_Turkey,team_name_Ukraine,team_name_United Arab Emirates,team_name_United States,team_name_Uruguay,team_name_Wales,team_name_West Germany,team_name_Yugoslavia,team_name_Zaire
0,1,0,1,0,1,17,0.058824,0.764706,2.705882,15,...,False,False,False,False,False,False,False,False,False,False
1,1,0,0,1,0,15,0.533333,1.600000,1.266667,17,...,False,False,False,False,False,False,False,False,False,False
2,1,0,0,1,0,16,0.500000,2.375000,2.187500,20,...,False,False,False,False,False,False,False,False,False,False
3,1,0,1,0,0,32,0.625000,2.625000,1.312500,19,...,False,False,False,False,False,False,False,False,False,False
4,1,0,1,0,0,5,0.200000,1.600000,2.400000,19,...,False,False,False,False,False,False,False,False,False,False


In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(
    X, Y, test_size=0.2, random_state=42, stratify=Y
)


In [8]:
log_model = LogisticRegression(max_iter=3000)
log_model.fit(X_train, Y_train)

log_pred = log_model.predict(X_test)

print("Logistic Regression")
print("Accuracy:", accuracy_score(Y_test, log_pred))
print("F1-score:", f1_score(Y_test, log_pred, average="macro"))

Logistic Regression
Accuracy: 0.6156716417910447
F1-score: 0.5651016213357897


In [9]:
scores = {}

for k in range(1, 16):
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train, Y_train)
    pred = knn.predict(X_test)
    scores[k] = f1_score(Y_test, pred, average="macro")

best_k = max(scores, key=scores.get)

print("Best K:", best_k)
print("Best F1-score:", scores[best_k])

Best K: 3
Best F1-score: 0.505992732999947


In [10]:
knn_model = KNeighborsClassifier(n_neighbors=best_k)
knn_model.fit(X_train, Y_train)

knn_pred = knn_model.predict(X_test)

print("KNN")
print("Accuracy:", accuracy_score(Y_test, knn_pred))
print("F1-score:", f1_score(Y_test, knn_pred, average="macro"))

KNN
Accuracy: 0.5373134328358209
F1-score: 0.505992732999947


In [11]:
print("Logistic Regression")
print(classification_report(Y_test, log_pred))

print("KNN")
print(classification_report(Y_test, knn_pred))

Logistic Regression
              precision    recall  f1-score   support

        Draw       0.48      0.31      0.38        52
        Loss       0.62      0.75      0.68       110
         Win       0.65      0.63      0.64       106

    accuracy                           0.62       268
   macro avg       0.59      0.56      0.57       268
weighted avg       0.61      0.62      0.60       268

KNN
              precision    recall  f1-score   support

        Draw       0.34      0.37      0.35        52
        Loss       0.59      0.62      0.60       110
         Win       0.59      0.54      0.56       106

    accuracy                           0.54       268
   macro avg       0.51      0.51      0.51       268
weighted avg       0.54      0.54      0.54       268

